# Yield Curve Prophet

## Why This Matters

The yield curve is the single most watched indicator in fixed income. Its shape dictates portfolio positioning, relative value trades, and macro regime classification. Every basis point move in the 2Y or 10Y triggers billions in hedging flow.

Most yield forecasting on Wall Street is still discretionary - PMs read the dot plot, watch payrolls, and make a call. This notebook asks whether a machine can do better than a coin flip at predicting the *direction* of weekly yield moves.

We build three classifiers - one for the 2Y (front-end, Fed-sensitive), one for the 10Y (term premium, growth expectations), and one for the 2s10s spread (curve shape, recession signal). Each model sees 60 trading days of macro history and predicts whether the next 5 trading days will move up or down.

**The ensemble (LSTM + XGBoost) lets us answer two questions:**
1. Does temporal sequence matter, or is today's snapshot sufficient? (LSTM vs XGBoost)
2. Can we beat 50%? By how much, and in which regimes?

All data is free via the FRED API. No Bloomberg required.

In [ ]:
import os
import warnings
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

import xgboost as xgb
import optuna
import shap
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, confusion_matrix,
    classification_report, roc_curve, calibration_curve,
)
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

FRED_API_KEY = os.environ["FRED_API_KEY"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch device: {DEVICE}")
print(f"Setup complete.")

In [ ]:
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

def fetch_fred_series(series_id: str, start: str = "2004-01-01",
                      end: str = "2025-12-31") -> pd.Series:
    """Pull a single FRED series and return as a pandas Series with datetime index."""
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start,
        "observation_end": end,
    }
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    data = resp.json()["observations"]
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    series = df.set_index("date")["value"].dropna()
    series.name = series_id
    time.sleep(0.1)
    return series


FRED_SERIES = [
    ("DGS2", "2Y Treasury Yield", "daily"),
    ("DGS5", "5Y Treasury Yield", "daily"),
    ("DGS10", "10Y Treasury Yield", "daily"),
    ("DGS30", "30Y Treasury Yield", "daily"),
    ("DFF", "Fed Funds Effective Rate", "daily"),
    ("T10YIE", "10Y Breakeven Inflation", "daily"),
    ("T5YIFR", "5Y5Y Forward Inflation", "daily"),
    ("UNRATE", "Unemployment Rate", "monthly"),
    ("ICSA", "Initial Jobless Claims", "weekly"),
    ("INDPRO", "Industrial Production Index", "monthly"),
    ("UMCSENT", "Consumer Sentiment", "monthly"),
    ("VIXCLS", "CBOE VIX", "daily"),
    ("DTWEXBGS", "Trade-Weighted Dollar Index", "daily"),
    ("TEDRATE", "TED Spread", "daily"),
    ("BAMLH0A0HYM2", "HY OAS", "daily"),
]

TRAIN_END = "2018-12-31"
VAL_END = "2020-12-31"

LOOKBACK = 60
HORIZON = 5
TARGETS = ["dgs2_dir", "dgs10_dir", "spread_2s10s_dir"]
TARGET_LABELS = {
    "dgs2_dir": "2Y Yield Direction",
    "dgs10_dir": "10Y Yield Direction",
    "spread_2s10s_dir": "2s10s Spread Direction",
}

print(f"Configured {len(FRED_SERIES)} FRED series")
print(f"Train: 2005 - {TRAIN_END}")
print(f"Val: {TRAIN_END} - {VAL_END}")
print(f"Test: {VAL_END} - 2025")

## 2. Data Collection

Pull all FRED series, align to a common daily trading calendar, and forward-fill lower-frequency series (monthly, weekly) to daily.

In [ ]:
print("Pulling FRED data...")
print("=" * 50)

raw_series = {}
for series_id, desc, freq in FRED_SERIES:
    try:
        s = fetch_fred_series(series_id)
        raw_series[series_id] = s
        print(f"  {series_id:20s} | {desc:35s} | {len(s):,} obs | {s.index[0].date()} to {s.index[-1].date()}")
    except Exception as e:
        print(f"  {series_id:20s} | FAILED: {e}")

print(f"\nPulled {len(raw_series)} / {len(FRED_SERIES)} series successfully.")

In [ ]:
trading_days = raw_series["DGS10"].index

df = pd.DataFrame(index=trading_days)
for series_id, s in raw_series.items():
    df[series_id] = s.reindex(df.index)

df = df.ffill()
df = df.loc["2005-01-01":]
df = df.dropna()

print(f"Aligned dataset: {df.shape[0]:,} trading days x {df.shape[1]} series")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nMissing values per series:")
print(df.isnull().sum())

In [ ]:
assert df.shape[0] > 4000, f"Expected ~5000 trading days, got {df.shape[0]}"
assert df.isnull().sum().sum() == 0, "No NaNs should remain after ffill + dropna"
assert "DGS2" in df.columns and "DGS10" in df.columns, "Must have 2Y and 10Y yields"

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Raw FRED Series: Yield Curve Inputs", fontsize=14, fontweight="bold")

ax = axes[0, 0]
for tenor in ["DGS2", "DGS5", "DGS10", "DGS30"]:
    ax.plot(df.index, df[tenor], label=tenor, linewidth=1)
ax.set_title("Treasury Yields")
ax.set_ylabel("Yield (%)")
ax.legend()

ax = axes[0, 1]
ax.plot(df.index, df["DFF"], color="darkred", linewidth=1)
ax.set_title("Fed Funds Rate")
ax.set_ylabel("Rate (%)")

ax = axes[1, 0]
ax.plot(df.index, df["VIXCLS"], color="purple", linewidth=0.8)
ax.set_title("VIX")
ax.set_ylabel("Level")

ax = axes[1, 1]
if "BAMLH0A0HYM2" in df.columns:
    ax.plot(df.index, df["BAMLH0A0HYM2"], color="orange", linewidth=1)
ax.set_title("HY OAS (bps)")
ax.set_ylabel("OAS")

plt.tight_layout()
plt.savefig("raw_fred_series.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: raw_fred_series.png")

## 3. Feature Engineering

For each raw series, compute rolling changes (5d, 21d, 63d), realized volatility (21d), z-scores (vs 252d rolling mean), and momentum signals. Also derive curve spreads (2s10s, 5s30s, real 10Y).

Total features: ~80-100 after engineering.

In [ ]:
df["spread_2s10s"] = df["DGS10"] - df["DGS2"]
df["spread_5s30s"] = df["DGS30"] - df["DGS5"]
df["real_10y"] = df["DGS10"] - df["T10YIE"]

print("Derived series added: spread_2s10s, spread_5s30s, real_10y")
print(f"Current 2s10s: {df['spread_2s10s'].iloc[-1]:.2f}%")
print(f"Current real 10Y: {df['real_10y'].iloc[-1]:.2f}%")

In [ ]:
def engineer_features(df: pd.DataFrame, series_cols: list[str]) -> pd.DataFrame:
    """
    Engineer rolling features for each series column.
    Returns a new DataFrame with original + engineered columns.
    """
    feat = df.copy()

    for col in series_cols:
        s = feat[col]

        # Rolling changes (absolute, in same units as original)
        feat[f"{col}_chg5d"] = s.diff(5)
        feat[f"{col}_chg21d"] = s.diff(21)
        feat[f"{col}_chg63d"] = s.diff(63)

        # Realized volatility (21-day rolling std of daily changes)
        daily_chg = s.diff(1)
        feat[f"{col}_vol21d"] = daily_chg.rolling(21).std()

        # Z-score vs 252-day rolling stats
        roll_mean = s.rolling(252).mean()
        roll_std = s.rolling(252).std()
        feat[f"{col}_zscore"] = (s - roll_mean) / roll_std.replace(0, np.nan)

        # Momentum: 5d MA vs 21d MA (1 = bullish, 0 = bearish)
        ma5 = s.rolling(5).mean()
        ma21 = s.rolling(21).mean()
        feat[f"{col}_momentum"] = (ma5 > ma21).astype(int)

    return feat


feature_cols = [sid for sid, _, _ in FRED_SERIES] + ["spread_2s10s", "spread_5s30s", "real_10y"]
df_feat = engineer_features(df, feature_cols)

df_feat = df_feat.dropna()

print(f"Feature matrix: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns")
print(f"Date range after engineering: {df_feat.index[0].date()} to {df_feat.index[-1].date()}")
print(f"\nSample feature names (first 20):")
for c in df_feat.columns[:20]:
    print(f"  {c}")
print(f"  ... and {len(df_feat.columns) - 20} more")

## 4. Target Construction

Create binary labels for 5-day forward yield/spread direction. Drop flat moves (exactly 0 change). Check class balance.

In [ ]:
df_feat["dgs2_fwd5d"] = df_feat["DGS2"].shift(-HORIZON) - df_feat["DGS2"]
df_feat["dgs10_fwd5d"] = df_feat["DGS10"].shift(-HORIZON) - df_feat["DGS10"]
df_feat["spread_2s10s_fwd5d"] = df_feat["spread_2s10s"].shift(-HORIZON) - df_feat["spread_2s10s"]

df_feat["dgs2_dir"] = (df_feat["dgs2_fwd5d"] > 0).astype(int)
df_feat["dgs10_dir"] = (df_feat["dgs10_fwd5d"] > 0).astype(int)
df_feat["spread_2s10s_dir"] = (df_feat["spread_2s10s_fwd5d"] > 0).astype(int)

for target in TARGETS:
    fwd_col = target.replace("_dir", "_fwd5d")
    zero_mask = df_feat[fwd_col] == 0
    n_zeros = zero_mask.sum()
    if n_zeros > 0:
        print(f"Dropping {n_zeros} flat moves for {target}")
        df_feat = df_feat[~zero_mask]

df_feat = df_feat.dropna(subset=TARGETS)

print("\nClass balance (% label=1):")
for target in TARGETS:
    pct = df_feat[target].mean() * 100
    print(f"  {TARGET_LABELS[target]:30s}: {pct:.1f}% up | {100-pct:.1f}% down | n={len(df_feat[target]):,}")

## 5. Train / Validation / Test Split

Walk-forward temporal split. No random shuffling. Rolling standardization uses only past data to prevent future leakage.

| Split | Period | Purpose |
|-------|--------|---------|
| Train | 2005-2018 | Model training |
| Validation | 2019-2020 | Hyperparameter tuning, early stopping, ensemble weights |
| Test | 2021-2025 | Final evaluation (all reported metrics) |

In [ ]:
exclude_suffixes = ["_fwd5d", "_dir"]
exclude_cols = [sid for sid, _, _ in FRED_SERIES] + ["spread_2s10s", "spread_5s30s", "real_10y"]
feature_names = [c for c in df_feat.columns
                 if c not in exclude_cols
                 and not any(c.endswith(s) for s in exclude_suffixes)]

print(f"Feature count: {len(feature_names)}")

train_mask = df_feat.index <= TRAIN_END
val_mask = (df_feat.index > TRAIN_END) & (df_feat.index <= VAL_END)
test_mask = df_feat.index > VAL_END

X_train_raw = df_feat.loc[train_mask, feature_names]
X_val_raw = df_feat.loc[val_mask, feature_names]
X_test_raw = df_feat.loc[test_mask, feature_names]

y_train = df_feat.loc[train_mask, TARGETS]
y_val = df_feat.loc[val_mask, TARGETS]
y_test = df_feat.loc[test_mask, TARGETS]

print(f"\nSplit sizes:")
print(f"  Train: {len(X_train_raw):,} ({X_train_raw.index[0].date()} to {X_train_raw.index[-1].date()})")
print(f"  Val:   {len(X_val_raw):,} ({X_val_raw.index[0].date()} to {X_val_raw.index[-1].date()})")
print(f"  Test:  {len(X_test_raw):,} ({X_test_raw.index[0].date()} to {X_test_raw.index[-1].date()})")

scaler = StandardScaler()
scaler.fit(X_train_raw)

X_train = pd.DataFrame(scaler.transform(X_train_raw), index=X_train_raw.index, columns=feature_names)
X_val = pd.DataFrame(scaler.transform(X_val_raw), index=X_val_raw.index, columns=feature_names)
X_test = pd.DataFrame(scaler.transform(X_test_raw), index=X_test_raw.index, columns=feature_names)

print(f"\nStandardized. Train mean ~0: {X_train.mean().mean():.4f}, std ~1: {X_train.std().mean():.4f}")

## 6. XGBoost Baseline

One XGBClassifier per target, tuned with Optuna (100 trials, ROC-AUC objective). This is the "does temporal structure matter?" benchmark - XGBoost sees only today's feature snapshot, not the sequence.

SHAP values reveal which macro features drive predictions.

In [ ]:
def tune_xgboost(X_tr: pd.DataFrame, y_tr: pd.Series,
                 X_v: pd.DataFrame, y_v: pd.Series,
                 n_trials: int = 100) -> dict:
    """Tune XGBoost hyperparameters with Optuna."""
    def objective(trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "eval_metric": "auc",
            "random_state": 42,
        }
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)
        y_prob = model.predict_proba(X_v)[:, 1]
        return roc_auc_score(y_v, y_prob)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params


xgb_models = {}
xgb_val_probs = {}
xgb_test_probs = {}

print("Training XGBoost models...")
print("=" * 60)

for target in TARGETS:
    print(f"\n--- {TARGET_LABELS[target]} ---")

    best_params = tune_xgboost(X_train, y_train[target], X_val, y_val[target], n_trials=100)
    best_params["eval_metric"] = "auc"
    best_params["random_state"] = 42
    print(f"Best params: max_depth={best_params['max_depth']}, lr={best_params['learning_rate']:.3f}, n_est={best_params['n_estimators']}")

    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train[target], eval_set=[(X_val, y_val[target])], verbose=False)

    xgb_val_probs[target] = model.predict_proba(X_val.values)[:, 1]
    xgb_test_probs[target] = model.predict_proba(X_test.values)[:, 1]
    xgb_models[target] = model

    val_acc = accuracy_score(y_val[target], (xgb_val_probs[target] > 0.5).astype(int))
    test_acc = accuracy_score(y_test[target], (xgb_test_probs[target] > 0.5).astype(int))
    test_auc = roc_auc_score(y_test[target], xgb_test_probs[target])
    print(f"Val acc: {val_acc:.1%} | Test acc: {test_acc:.1%} | Test AUC: {test_auc:.3f}")

print("\nXGBoost training complete.")

In [ ]:
print("Computing SHAP values...")

fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle("XGBoost Feature Importance (SHAP)", fontsize=14, fontweight="bold")

for idx, target in enumerate(TARGETS):
    explainer = shap.TreeExplainer(xgb_models[target])
    shap_values = explainer.shap_values(X_test)

    ax = axes[idx]
    shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=15,
                      show=False, ax=ax)
    ax.set_title(TARGET_LABELS[target])

plt.tight_layout()
plt.savefig("xgb_shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: xgb_shap_importance.png")

## 7. LSTM Neural Network

Two-layer LSTM that ingests 60-day sequences of macro features. Unlike XGBoost, which sees only today's snapshot, the LSTM learns temporal patterns - momentum shifts, volatility clustering, and regime transitions that play out over weeks.

Architecture: LSTM(128) -> LSTM(64) -> Dense(64) -> Sigmoid

In [ ]:
class YieldDataset(Dataset):
    """Sliding window dataset for LSTM input."""
    def __init__(self, X: np.ndarray, y: np.ndarray, lookback: int = 60):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.lookback = lookback

    def __len__(self):
        return len(self.X) - self.lookback

    def __getitem__(self, idx):
        x_seq = self.X[idx:idx + self.lookback]
        y_val = self.y[idx + self.lookback]
        return x_seq, y_val


class YieldLSTM(nn.Module):
    """Two-layer LSTM classifier for yield direction prediction."""
    def __init__(self, input_size: int, hidden1: int = 128, hidden2: int = 64,
                 dropout: float = 0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden1, batch_first=True, dropout=dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out, _ = self.lstm1(x)
        out, _ = self.lstm2(out)
        out = out[:, -1, :]
        return self.head(out).squeeze(-1)


print(f"LSTM architecture defined.")
print(f"Input features: {len(feature_names)}")
print(f"Lookback window: {LOOKBACK} days")
print(f"Device: {DEVICE}")

In [ ]:
def train_lstm(X_tr: np.ndarray, y_tr: np.ndarray,
               X_v: np.ndarray, y_v: np.ndarray,
               input_size: int, target_name: str,
               max_epochs: int = 100, patience: int = 10,
               batch_size: int = 64, lr: float = 1e-3) -> nn.Module:
    """Train LSTM with early stopping on validation loss."""
    train_ds = YieldDataset(X_tr, y_tr, LOOKBACK)
    val_ds = YieldDataset(X_v, y_v, LOOKBACK)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = YieldLSTM(input_size).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5
    )

    best_val_loss = float("inf")
    best_state = None
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * len(xb)
        val_loss /= len(val_ds)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.1e}")

        if wait >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    return model


def predict_lstm(model: nn.Module, X: np.ndarray) -> np.ndarray:
    """Generate predictions from LSTM model on a full array."""
    ds = YieldDataset(X, np.zeros(len(X)), LOOKBACK)
    dl = DataLoader(ds, batch_size=256, shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(DEVICE)
            pred = model(xb)
            preds.append(pred.cpu().numpy())
    return np.concatenate(preds)

In [ ]:
X_all = pd.concat([X_train, X_val, X_test]).values
y_all_dict = {t: pd.concat([y_train[t], y_val[t], y_test[t]]).values for t in TARGETS}

n_train = len(X_train)
n_val = len(X_val)
n_test = len(X_test)

lstm_models = {}
lstm_val_probs = {}
lstm_test_probs = {}

print("Training LSTM models...")
print("=" * 60)

for target in TARGETS:
    print(f"\n--- {TARGET_LABELS[target]} ---")

    X_tr_seq = X_all[:n_train + LOOKBACK]
    y_tr_seq = y_all_dict[target][:n_train + LOOKBACK]

    X_v_seq = X_all[:n_train + n_val + LOOKBACK]
    y_v_seq = y_all_dict[target][:n_train + n_val + LOOKBACK]

    model = train_lstm(
        X_tr_seq, y_tr_seq,
        X_v_seq, y_v_seq,
        input_size=len(feature_names),
        target_name=target,
    )
    lstm_models[target] = model

    val_preds = predict_lstm(model, X_all[:n_train + n_val])
    lstm_val_probs[target] = val_preds[n_train - LOOKBACK:]

    test_preds = predict_lstm(model, X_all)
    lstm_test_probs[target] = test_preds[n_train + n_val - LOOKBACK:]

    lstm_val_probs[target] = lstm_val_probs[target][:n_val]
    lstm_test_probs[target] = lstm_test_probs[target][:n_test]

    val_acc = accuracy_score(y_val[target].values, (lstm_val_probs[target] > 0.5).astype(int))
    test_acc = accuracy_score(y_test[target].values[:len(lstm_test_probs[target])],
                              (lstm_test_probs[target] > 0.5).astype(int))
    print(f"  Val acc: {val_acc:.1%} | Test acc: {test_acc:.1%}")

print("\nLSTM training complete.")

## 8. Ensemble

Weighted average of XGBoost and LSTM probabilities. Weight optimized per target on validation set accuracy via grid search.

`p_ensemble = w * p_xgb + (1-w) * p_lstm`

In [ ]:
ensemble_weights = {}
ensemble_test_probs = {}

print("Optimizing ensemble weights...")
print("=" * 60)

for target in TARGETS:
    best_w = 0.5
    best_acc = 0

    n_common_val = min(len(xgb_val_probs[target]), len(lstm_val_probs[target]))
    xgb_vp = xgb_val_probs[target][:n_common_val]
    lstm_vp = lstm_val_probs[target][:n_common_val]
    y_v = y_val[target].values[:n_common_val]

    for w in np.arange(0, 1.05, 0.05):
        blended = w * xgb_vp + (1 - w) * lstm_vp
        acc = accuracy_score(y_v, (blended > 0.5).astype(int))
        if acc > best_acc:
            best_acc = acc
            best_w = w

    ensemble_weights[target] = best_w

    n_common_test = min(len(xgb_test_probs[target]), len(lstm_test_probs[target]))
    ensemble_test_probs[target] = (
        best_w * xgb_test_probs[target][:n_common_test]
        + (1 - best_w) * lstm_test_probs[target][:n_common_test]
    )

    test_acc = accuracy_score(
        y_test[target].values[:n_common_test],
        (ensemble_test_probs[target] > 0.5).astype(int)
    )
    print(f"  {TARGET_LABELS[target]:30s} | w_xgb={best_w:.2f} | Val acc={best_acc:.1%} | Test acc={test_acc:.1%}")

print("\nEnsemble optimization complete.")